# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_step,
    run_worm
)
import jax
import jax.numpy as jnp
import numpy as np
from typing import Tuple, Dict
from jax import Array
from jax.typing import ArrayLike
from functools import partial

In [2]:
length = 11  # 5
width = 11  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

base_key = jax.random.PRNGKey(42)
initial_error = em_lindblad.generate_random_error(base_key)
initial_error_mod_2 = jnp.mod(initial_error, 2)
initial_error_mod_p = jnp.mod(initial_error, p)
# Here we consider the full syndrome including the plaquette
# we usually remove because of the constraint as this simplified the
# coding of the worm algorithm. In fact, in this the syndromes will
# always be annihilated in pairs, and the total number of syndromes is
# always even as one can check numerically.
syndrome = moebius_code.get_plaquette_syndrome(initial_error)
syndrome_mod_2 = jnp.mod(syndrome, 2)
syndrome_mod_p = jnp.mod(syndrome, p)
num_plaquette, num_edges = h_x.shape

In [3]:
# worm_step_partial = partial(
#     worm_step, h_error_mod_p=h_z_mod_p, h_mod_p=h_x_mod_p, error_model=em_lindblad)
base_key = jax.random.PRNGKey(144)
max_steps = 10000
initial_worm_state = {}
worm_error = jnp.vstack(
    (initial_error_mod_2, initial_error_mod_p))
# initial_worm_state["worm_error"] = jnp.vstack(
#     (initial_error_mod_2, initial_error_mod_p))
initial_head = 18
initial_worm_state["head"] = initial_head
initial_worm_state["tail"] = initial_head
initial_worm_state["worm_success"] = False
# initial_worm_state["h_error_mod_p"] = h_z_mod_p
# initial_worm_state["h_mod_p"] = h_x_mod_p
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0
# initial_worm_state["key"] = base_key

# new_worm_state = initial_worm_state.copy()

# # jit_worm_step_partial = jax.jit(worm_step_partial)

# new_worm_state, _ = jax.lax.scan(
#     worm_step_partial, initial_worm_state, jnp.arange(max_steps))

new_worm_state = run_worm(
    worm_error,
    base_key,
    initial_worm_state,
    h_z_mod_p,
    h_x_mod_p,
    em_lindblad,
    max_steps
)

print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"]))

Number of accepted moves: 66
Number of attempted moves: 1490


In [4]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

18
18


In [6]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][0, :] - worm_error[0, :], 2))))

1


In [7]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][1, :] - worm_error[1, :], p))))

2


In [8]:
new_syndrome_mod_2 = jnp.mod(h_x_mod_2 @ new_worm_state["worm_error"][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [9]:
new_syndrome_mod_p = jnp.mod(h_x_mod_p @ new_worm_state["worm_error"][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)